# Phase C: SLM-LLM Supply Chain Relation Extractor

**목적**: 뉴스 399만건에서 공급망 관계를 자동 추출하여 KG 엣지 자동 생성

**구조** (가이드.docx Section 3.2 기반):
```
Phase 1: SLM (Qwen2.5-3B) — 대량 후보 생성 (빠르고 저비용)
Phase 2: LLM (Llama-3.1-8B) — Reflection 검증 (정밀하고 논리적)
```

**필요 환경**: Colab GPU (T4: Qwen만, A100: Qwen+Llama)

**입력**: `risk_events.parquet` (title, url, entity_ids)
**출력**: `extracted_edges.parquet` (자동 추출된 공급망 관계)

---

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/nabi_hyoghaw'  # ← 본인 경로로 수정
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q transformers torch accelerate bitsandbytes pandas pyarrow tqdm

import torch
print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f'  {gpu}, {mem:.1f} GB')
    if mem < 14:
        print('  ⚠️ T4 감지: Qwen2.5-3B만 사용 (4-bit 양자화)')
        USE_LLM_VERIFIER = False
    else:
        print('  ✅ A100/L4 감지: Qwen + Llama 모두 사용 가능')
        USE_LLM_VERIFIER = True

## 1. Universe + Ontology 로드

In [ ]:
import pandas as pd
import yaml
from pathlib import Path

# Universe (company_id → name 매핑)
uni = pd.read_csv('data/universe/univers_final.csv')
COMPANY_MAP = dict(zip(uni['company_id'], uni['canonical_name']))
COMPANY_NAMES = set(uni['canonical_name'].str.lower())

# Aliases
with open('configs/company_aliases.yaml', 'r', encoding='utf-8') as f:
    aliases_data = yaml.safe_load(f)
ALIAS_TO_CID = {}
for cid, info in aliases_data.items():
    for alias in info.get('aliases', []):
        ALIAS_TO_CID[alias.lower()] = cid
    ALIAS_TO_CID[info.get('canonical_name', '').lower()] = cid

# Ontology (허용된 관계 유형)
ONTOLOGY = ['SUPPLIES', 'BUYS_FROM', 'PARTNERS_WITH', 'COMPETES_WITH', 'SUBSIDIARY_OF', 'OWNS']

print(f'Universe: {len(COMPANY_MAP)} companies, {len(ALIAS_TO_CID)} aliases')
print(f'Ontology: {ONTOLOGY}')

## 2. SLM (Qwen2.5-3B) 로드 — Phase 1: 후보 생성

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

SLM_MODEL = 'Qwen/Qwen2.5-3B-Instruct'

print(f'Loading SLM: {SLM_MODEL}...')
slm_tokenizer = AutoTokenizer.from_pretrained(SLM_MODEL, trust_remote_code=True)
slm_model = AutoModelForCausalLM.from_pretrained(
    SLM_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
slm_model.eval()
print('SLM loaded ✅')

## 3. Relation Extraction Prompt

In [ ]:
import json
import re

EXTRACTION_PROMPT = """You are a supply chain relation extractor for the EV battery industry.

Given a news headline, extract supply chain relationships as JSON.

Rules:
- Only extract relationships between known companies
- Use ONLY these relation types: {ontology}
- Return a JSON array of objects with: src, rel, dst, confidence (0-1)
- If no relationship found, return []
- confidence: 1.0 = explicitly stated, 0.7 = implied, 0.5 = speculative

Known companies: {companies}

Headline: "{headline}"

Output (JSON array only):"""

def build_prompt(headline, companies_subset):
    """Build extraction prompt with relevant companies only."""
    return EXTRACTION_PROMPT.format(
        ontology=', '.join(ONTOLOGY),
        companies=', '.join(companies_subset[:30]),  # top 30 relevant
        headline=headline[:300],
    )

def parse_slm_output(text):
    """Parse SLM JSON output, handling common errors."""
    # Find JSON array in output
    match = re.search(r'\[.*?\]', text, re.DOTALL)
    if not match:
        return []
    try:
        candidates = json.loads(match.group())
        if isinstance(candidates, list):
            return candidates
    except json.JSONDecodeError:
        pass
    return []

# 테스트
test_prompt = build_prompt(
    'CATL signs major battery supply deal with BMW for next-generation EVs',
    ['CATL', 'BMW', 'Tesla', 'LG Energy Solution', 'BYD']
)
print(test_prompt)

## 4. SLM Batch Extraction

In [ ]:
from tqdm.auto import tqdm
import numpy as np

def slm_extract_batch(headlines, slm_model, slm_tokenizer, batch_size=1):
    """
    SLM으로 뉴스 제목에서 관계 후보 추출.
    batch_size=1 (생성 모델은 보통 1개씩)
    """
    all_candidates = []

    for headline in tqdm(headlines, desc='SLM Extraction'):
        if not headline or len(headline) < 10:
            all_candidates.append([])
            continue

        # 관련 기업 찾기 (headline에 등장하는 기업)
        hl_lower = headline.lower()
        relevant = [name for alias, cid in ALIAS_TO_CID.items()
                     if alias in hl_lower
                     for name in [COMPANY_MAP.get(cid, cid)]]
        relevant = list(set(relevant))[:30]

        if len(relevant) < 2:
            all_candidates.append([])
            continue

        prompt = build_prompt(headline, relevant)
        messages = [{'role': 'user', 'content': prompt}]
        text = slm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = slm_tokenizer(text, return_tensors='pt').to(slm_model.device)

        with torch.no_grad():
            outputs = slm_model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.1,
                do_sample=False,
            )

        response = slm_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        candidates = parse_slm_output(response)
        all_candidates.append(candidates)

    return all_candidates

print('SLM extraction function ready')

## 5. 데이터 로드 + 필터 (2개 이상 기업 언급된 뉴스만)

In [ ]:
# risk_events에서 2개 이상 기업이 언급된 뉴스만 추출
df = pd.read_parquet('data/processed/risk_events.parquet',
                     columns=['event_id', 'title', 'entity_ids'])

# entity_ids가 2개 이상인 행만 (관계 추출 가능)
df['n_entities'] = df['entity_ids'].apply(
    lambda x: len(x) if hasattr(x, '__len__') else 0
)
multi = df[df['n_entities'] >= 2].copy()
print(f'Total events: {len(df):,}')
print(f'Multi-entity events: {len(multi):,} ({100*len(multi)/len(df):.1f}%)')

# 제목이 있는 것만
multi = multi[multi['title'].str.len() > 20]
print(f'With title: {len(multi):,}')

# 중복 제목 제거
multi = multi.drop_duplicates(subset='title')
print(f'Unique titles: {len(multi):,}')

In [ ]:
# 샘플로 먼저 테스트 (100개)
SAMPLE_SIZE = 100  # 전체: len(multi)
sample = multi.head(SAMPLE_SIZE)
print(f'Processing {len(sample)} headlines...')

candidates = slm_extract_batch(sample['title'].tolist(), slm_model, slm_tokenizer)

# 결과 확인
n_found = sum(1 for c in candidates if len(c) > 0)
total_rels = sum(len(c) for c in candidates)
print(f'\nResults: {n_found}/{len(sample)} headlines had relations ({total_rels} total)')

# 예시 출력
for i, (hl, cands) in enumerate(zip(sample['title'].tolist(), candidates)):
    if cands:
        print(f'\n[{i}] {hl[:100]}')
        for c in cands:
            print(f'  → {c}')

## 6. (A100만) LLM Verifier — Phase 2: Reflection 검증

In [ ]:
if USE_LLM_VERIFIER:
    LLM_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
    print(f'Loading LLM verifier: {LLM_MODEL}...')

    llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL,
        torch_dtype=torch.float16,
        device_map='auto',
    )
    llm_model.eval()
    print('LLM verifier loaded ✅')
else:
    print('⚠️ T4 GPU: LLM verifier skipped (confidence threshold로 대체)')

In [ ]:
VERIFY_PROMPT = """You are verifying a supply chain relationship extracted from a news headline.

Headline: "{headline}"
Extracted: {src} --{rel}--> {dst} (confidence: {conf})

Questions:
1. Does the headline actually support this relationship?
2. Is the direction correct? (src supplies/sells TO dst)
3. Is the relation type appropriate?

Answer with JSON: {{"valid": true/false, "corrected_rel": "...", "confidence": 0.0-1.0, "reason": "..."}}"""

def verify_with_llm(headline, candidate, llm_model, llm_tokenizer):
    """LLM으로 추출된 관계 검증 (Reflection)."""
    prompt = VERIFY_PROMPT.format(
        headline=headline[:300],
        src=candidate.get('src', ''),
        rel=candidate.get('rel', ''),
        dst=candidate.get('dst', ''),
        conf=candidate.get('confidence', 0),
    )
    messages = [{'role': 'user', 'content': prompt}]
    text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer(text, return_tensors='pt').to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(**inputs, max_new_tokens=150, temperature=0.1, do_sample=False)

    response = llm_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    try:
        match = re.search(r'\{.*?\}', response, re.DOTALL)
        if match:
            result = json.loads(match.group())
            return result
    except:
        pass
    return {'valid': False, 'confidence': 0}

print('LLM verification function ready')

## 7. 전체 실행 + 엣지 매핑

In [ ]:
def map_to_company_id(name):
    """회사 이름/alias → company_id 매핑."""
    if not name:
        return None
    nl = name.lower().strip()
    # 직접 매핑
    if nl in ALIAS_TO_CID:
        return ALIAS_TO_CID[nl]
    # 부분 매칭
    for alias, cid in ALIAS_TO_CID.items():
        if alias in nl or nl in alias:
            return cid
    return None

def candidates_to_edges(all_candidates, headlines, min_confidence=0.7):
    """SLM 후보 → seed_edges 형식 변환."""
    edges = []
    for headline, cands in zip(headlines, all_candidates):
        for c in cands:
            conf = c.get('confidence', 0)
            if conf < min_confidence:
                continue

            src_cid = map_to_company_id(c.get('src', ''))
            dst_cid = map_to_company_id(c.get('dst', ''))
            rel = c.get('rel', '').upper().replace(' ', '_')

            if not src_cid or not dst_cid:
                continue
            if src_cid == dst_cid:
                continue
            if rel not in ONTOLOGY:
                continue

            edges.append({
                'src_company_id': src_cid,
                'rel_type': rel,
                'dst_company_id': dst_cid,
                'confidence_plink': round(conf, 2),
                'strength': round(conf * 0.8, 2),
                'evidence': f'LLM-extracted: {headline[:150]}',
                'source': 'SLM_Qwen2.5-3B',
                'valid_from': '',
                'valid_to': '',
            })

    # Dedup
    seen = set()
    unique = []
    for e in edges:
        key = (e['src_company_id'], e['rel_type'], e['dst_company_id'])
        if key not in seen:
            seen.add(key)
            unique.append(e)

    return pd.DataFrame(unique)

extracted = candidates_to_edges(candidates, sample['title'].tolist())
print(f'Extracted edges: {len(extracted)}')
if len(extracted) > 0:
    print(extracted[['src_company_id', 'rel_type', 'dst_company_id', 'confidence_plink']].to_string())

## 8. 전체 데이터 실행 (선택)

⚠️ T4 기준 유니크 타이틀 수에 따라 수 시간 소요될 수 있습니다.
A100이면 ~1시간.

In [ ]:
# 전체 실행 (주석 해제 후 실행)
# FULL_RUN = True
FULL_RUN = False

if FULL_RUN:
    print(f'Full extraction: {len(multi):,} unique headlines')
    all_cands = slm_extract_batch(multi['title'].tolist(), slm_model, slm_tokenizer)
    all_edges = candidates_to_edges(all_cands, multi['title'].tolist())
    print(f'Total extracted edges: {len(all_edges)}')

    # 저장
    all_edges.to_parquet('data/processed/extracted_edges_slm.parquet', index=False)
    print('Saved: data/processed/extracted_edges_slm.parquet')
else:
    print('Set FULL_RUN = True to process all headlines')

## 9. 기존 seed_edges와 병합

In [ ]:
# 기존 seed_edges에 신규 엣지 추가 (중복 제거)
if FULL_RUN and len(all_edges) > 0:
    seed = pd.read_csv('data/seed/seed_edges.csv')
    existing_keys = set(zip(seed['src_company_id'], seed['rel_type'], seed['dst_company_id']))

    new_only = all_edges[
        ~all_edges.apply(lambda r: (r['src_company_id'], r['rel_type'], r['dst_company_id']) in existing_keys, axis=1)
    ]

    print(f'Existing edges: {len(seed)}')
    print(f'SLM extracted: {len(all_edges)}')
    print(f'Truly new: {len(new_only)}')

    if len(new_only) > 0:
        # 컬럼 맞추기
        for col in seed.columns:
            if col not in new_only.columns:
                new_only[col] = ''
        new_only = new_only[seed.columns]

        merged = pd.concat([seed, new_only], ignore_index=True)
        merged.to_csv('data/seed/seed_edges.csv', index=False, encoding='utf-8-sig')
        print(f'Merged: {len(merged)} total edges')
else:
    print('Run full extraction first (FULL_RUN = True)')